In [ ]:
### Task 1: Load Data with textFile
from pyspark import SparkContext


# Load the CSV file
sc = SparkContext("local[*]", "DataIO")
lines = sc.textFile("./sales_data.csv")


# Skip header line
header_one = lines.first()
lines = lines.filter(lambda line: line != header_one)

print(f"Header: {header_one}")
print(f"Data records: {lines.count()}")
print(f"First record: {lines.first()}")


### Task 2: Parse CSV Records
def parse_line(line):
    lines = line.split(",")
    return {"product_id": lines[0], "name": lines[1], "category": lines[2], "price": float(lines[3]),"quantity": int(lines[4])}

# Parse all records
parsed_lines = lines.map(parse_line)

# Show parsed data
for record in parsed_lines.collect():
    print(record)


### Task 3: Process and Save Results
revenue = parsed_lines.map(lambda r: f"{r['product_id']}, {r['name']}, {r['price']}, {r['quantity']}")

# Save to output directory
revenue.saveAsTextFile("output_revenue")



### Task 4: Load Multiple Files
#Load all CSV files using wildcard pattern
all_data_raw = sc.textFile("sales_data*.csv")

# Clean the data (Filter out headers from ALL files)
header_two = all_data_raw.first()
all_data = all_data_raw.filter(lambda line: line != header_two)


# Parse the multi-file data
parsed_all = all_data.map(parse_line)

# Calculate revenue
revenue_all = parsed_all.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

# YOUR CODE: Use coalesce(1) before saveAsTextFile
single_output_revenue = revenue_all.coalesce(1)

# Save to a new output directory
single_output_revenue.saveAsTextFile("final_single_revenue")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 23:47:37 WARN Utils: Your hostname, dYesgat, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 23:47:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 23:47:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Header: product_id,name,category,price,quantity


Data records: 7


First record: P001,Laptop,Electronics,999.99,5


{'product_id': 'P001', 'name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'quantity': 5}
{'product_id': 'P002', 'name': 'Mouse', 'category': 'Electronics', 'price': 29.99, 'quantity': 50}
{'product_id': 'P003', 'name': 'Desk', 'category': 'Furniture', 'price': 199.99, 'quantity': 10}
{'product_id': 'P004', 'name': 'Chair', 'category': 'Furniture', 'price': 149.99, 'quantity': 20}
{'product_id': 'P005', 'name': 'Monitor', 'category': 'Electronics', 'price': 299.99, 'quantity': 15}
{'product_id': 'P006', 'name': 'Keyboard', 'category': 'Electronics', 'price': 79.99, 'quantity': 30}
{'product_id': 'P007', 'name': 'Lamp', 'category': 'Furniture', 'price': 49.99, 'quantity': 25}
